# Notebook 2 — Input Size Experiment

The baseline was trained at 128×128. Before building anything more
complex, we need to answer one question: is 128×128 the right choice?

We train the exact same baseline model at three input sizes and
compare accuracy vs training time. The winner becomes the fixed
input size for all future notebooks.

Models tested: 64×64 | 128×128 | 224×224


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jessicali9530/stanford-dogs-dataset")

print("Path to dataset files:", path)

100%|██████████| 750M/750M [00:07<00:00, 108MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jessicali9530/stanford-dogs-dataset/versions/2


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import matplotlib.pyplot as plt
import pandas as pd
import time

In [ ]:
def load_data(img_size, batch_size=32):

    train = keras.utils.image_dataset_from_directory(
        path + "/images/Images",
        validation_split=0.2,
        subset="training",
        seed=42,
        labels='inferred',
        label_mode='int',
        batch_size=batch_size,
        image_size=(img_size, img_size)
    )

    val = keras.utils.image_dataset_from_directory(
        path + "/images/Images",
        validation_split=0.2,
        subset="validation",
        seed=42,
        labels='inferred',
        label_mode='int',
        batch_size=batch_size,
        image_size=(img_size, img_size)
    )
    def norm(image, label):
        return tf.cast(image/255., tf.float32), label

    return train.map(norm).prefetch(tf.data.AUTOTUNE), \
           val.map(norm).prefetch(tf.data.AUTOTUNE)

In [ ]:
def baseline_inputs(input_size):
    model=Sequential([
        #block 1
        #param: (3x3x3=1)x32=896
        Conv2D(32,(3,3),padding='valid',activation='relu',input_shape=(input_size,input_size,3)),
        MaxPooling2D((2,2)),

        #block 2
        #param:
        Conv2D(64,(3,3),padding='valid',activation='relu'),
        MaxPooling2D((2,2)),

        #block 3
        #param:
        Conv2D(128,(3,3),padding='valid',activation='relu'),
        MaxPooling2D((2,2)),

        Flatten(),
        Dense(128,activation='relu'),
        Dense(120,activation='softmax')

    ])
    model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

    return model


In [ ]:
#EXPERIMENT

input_size=[64,128,224]
results={}

for size in input_size:
  print(f'\n Training at {size}x{size} ')
  tf.keras.backend.clear_session()

  train_data,val_data =load_data(size)
  model=baseline_inputs(size)

  start=time.time()
  history=model.fit(train_data,validation_data=val_data,epochs=10,verbose=1)
  end=time.time() - start

  results[size]={
      'val_acc':max(history.history['val_accuracy']), #getting max val acc
      'train_acc':max(history.history['accuracy']),#getting max val loss
      'history':history,
      'params':model.count_params(),
      'time_mins':round(end/60,1)
  }
  print(f"Done — val acc: {results[size]['val_acc']:.4f} "
          f"in {results[size]['time_mins']} mins")



 Training at 64x64 
Found 20580 files belonging to 120 classes.
Using 16464 files for training.
Found 20580 files belonging to 120 classes.
Using 4116 files for validation.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 30s 47ms/step - accuracy: 0.0176 - loss: 4.7097 - val_accuracy: 0.0309 - val_loss: 4.5600
Epoch 2/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.0453 - loss: 4.3515 - val_accuracy: 0.0517 - val_loss: 4.2945
Epoch 3/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - accuracy: 0.0712 - loss: 4.1232 - val_accuracy: 0.0656 - val_loss: 4.1730
Epoch 4/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.0955 - loss: 3.9324 - val_accuracy: 0.0787 - val_loss: 4.1387
Epoch 5/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 21s 39ms/step - accuracy: 0.1323 - loss: 3.7308 - val_accuracy: 0.0887 - val_loss: 4.1345
Epoch 6/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.1655 - loss: 3.5076 - val_accuracy: 0.0921 - val_loss: 4.1627
Epoch 7/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.2069 - loss: 3.2766 - val_accuracy: 0.0938 - val_loss: 4.2574
Epoch 8/10
515/515 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.2544 - loss: 3.0197 - 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, size in enumerate(input_size):
    h = results[size]['history']
    axes[i].plot(h.history['accuracy'], label='train', color='royalblue')
    axes[i].plot(h.history['val_acc'], label='val',
                 color='tomato', linestyle='--')
    axes[i].set_title(f'{size}×{size}\nval acc: '
                      f'{results[size]["val_acc"]:.2%} | '
                      f'{results[size]["time_mins"]} mins')
    axes[i].legend()
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Accuracy')

plt.suptitle('Input Size Comparison — Same Model, Different Input',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Notebook 2 Results —> Input Size Experiment

| Input Size | Val Accuracy | Train Accuracy | Training Time | Overfitting |
|---|---|---|---|---|
| 64×64   | 9.52% | 36.44% | 3.6 mins | moderate |
| 128×128 | 7.39% | 71.39% | 5.7 mins | severe |
| 224×224 | 4.62% | 96.44% | 6.9 mins | extreme |

## What I observed

Counterintuitively, 64×64 gave the best val accuracy under the baseline CNN architecture,not because
it's a better input size, but because the smaller spatial dimensions
act as a natural regularizer in this case. Less detail means less for the model
to memorize.

224×224 is the worst train accuracy hits 96% while val stays at
4.6%. The model has enormous capacity to memorize with high resolution
images but no regularization to stop it.

128×128 sits in between but still overfits badly.

The real lesson here is not about input size it's that without
regularization, more detail = more overfitting. This motivates
everything in notebooks 3 and 4.

###*Decision: use 128×128 for all future notebooks standard choice, well understood, middle ground*

-------------------------------   
Baseline to beat: 9.52% val accuracy (64×64 result)
-------------------------------

## Reproducibility note

| Run | Model | Val Accuracy |
|---|---|---|
| Notebook 1 | Baseline (128×128) | 5.27% |
| Notebook 2 | Same model retrained | 9.52% |

Same code, different result  this is training variance (random weight initialization + batch shuffling).
Fixed from notebook 3 onwards with seed=42.